In [1]:
#task 4

In [2]:
import sys, json
sys.path.append('..')

from ner.utils import load_jsonl, save_responses_to_jsonl
from ner.evaluation import evaluate_ner_strict, evaluate_ner_relaxed

VALIDATION_PATH = '../data/plain/validation.jsonl'
OUTPUT_PATH = '../results/llm/llm_validation.jsonl'
N = 10   # número de documentos que pasamos al chat

LABELS = [
    'PERSON_NAME', 'ID_CARD_NUMBER', 'ADDRESS',
    'REGISTRATION_NUMBER', 'DATE', 'AMOUNT', 'ORGANIZATION'
]

samples = load_jsonl(VALIDATION_PATH, n=N)

In [3]:
#celda para generar cada documento
INSTRUCCIONES = f"""Eres un sistema de reconocimiento de entidades nombradas (NER).
Extrae del texto todas las entidades de estos tipos: {', '.join(LABELS)}.

Reglas:
- Copia cada entidad EXACTAMENTE como aparece en el texto, sin corregir ni resumir.
- Incluye todas las menciones, también las repetidas, en el orden en que aparecen.
- No inventes entidades.
- Responde SOLO con un JSON válido, sin explicaciones, con esta forma:
[{{"text": "...", "label": "..."}}]
"""

def make_prompt(sample):
    return INSTRUCCIONES + "\nTEXTO:\n" + sample["text"]

print(samples[0]["id"])
print(make_prompt(samples[0]))

test-0001
Eres un sistema de reconocimiento de entidades nombradas (NER).
Extrae del texto todas las entidades de estos tipos: PERSON_NAME, ID_CARD_NUMBER, ADDRESS, REGISTRATION_NUMBER, DATE, AMOUNT, ORGANIZATION.

Reglas:
- Copia cada entidad EXACTAMENTE como aparece en el texto, sin corregir ni resumir.
- Incluye todas las menciones, también las repetidas, en el orden en que aparecen.
- No inventes entidades.
- Responde SOLO con un JSON válido, sin explicaciones, con esta forma:
[{"text": "...", "label": "..."}]

TEXTO:
Servicio de Procedimientos Públicos de la Región Central informa de que Patricia Giménez Domínguez, con documento de identidad 12082063W y domicilio habitual en Avenida de Aragón, 102, 2-B, 17001 Girona, presentó el 2018-04-09 una solicitud relacionada con la inscripción en un registro administrativo sectorial. La petición se registró al sistema con el número ASS-2018-43226, y el justificante de entrada fue remitido a Patricia Giménez Domínguez mediante el canal señal

In [4]:
LLM_ANSWERS = {
    "test-0001": r"""[
{"text": "Servicio de Procedimientos Públicos de la Región Central", "label": "ORGANIZATION"},
{"text": "Patricia Giménez Domínguez", "label": "PERSON_NAME"},
{"text": "12082063W", "label": "ID_CARD_NUMBER"},
{"text": "Avenida de Aragón, 102, 2-B, 17001 Girona", "label": "ADDRESS"},
{"text": "2018-04-09", "label": "DATE"},
{"text": "ASS-2018-43226", "label": "REGISTRATION_NUMBER"},
{"text": "Patricia Giménez Domínguez", "label": "PERSON_NAME"},
{"text": "Tomás Romero Domínguez", "label": "PERSON_NAME"},
{"text": "12082200R", "label": "ID_CARD_NUMBER"},
{"text": "Calle de la Estación, 38, 6-2a, 08221 Terrassa", "label": "ADDRESS"},
{"text": "Instituto de Coordinación Territorial del Área Metropolitana", "label": "ORGANIZATION"},
{"text": "15 de abril de 2018", "label": "DATE"},
{"text": "Tomás Romero Domínguez", "label": "PERSON_NAME"},
{"text": "Patricia Giménez Domínguez", "label": "PERSON_NAME"},
{"text": "ASS-2018-43226", "label": "REGISTRATION_NUMBER"},
{"text": "12082063W", "label": "ID_CARD_NUMBER"},
{"text": "21/04/2018", "label": "DATE"},
{"text": "4.416,13 euros", "label": "AMOUNT"},
{"text": "Unidad de Atención Ciudadana de la Comarca Interior", "label": "ORGANIZATION"},
{"text": "1.107,16 EUR", "label": "AMOUNT"},
{"text": "5.523,29 €", "label": "AMOUNT"},
{"text": "2018-04-27", "label": "DATE"},
{"text": "ASS-2018-43226", "label": "REGISTRATION_NUMBER"},
{"text": "3 de mayo de 2018", "label": "DATE"},
{"text": "Avenida de Aragón, 102, 2-B, 17001 Girona", "label": "ADDRESS"},
{"text": "Tomás Romero Domínguez", "label": "PERSON_NAME"},
{"text": "12082200R", "label": "ID_CARD_NUMBER"},
{"text": "09/05/2018", "label": "DATE"},
{"text": "ENT-2018-43263", "label": "REGISTRATION_NUMBER"},
{"text": "Inés Suárez Giménez", "label": "PERSON_NAME"},
{"text": "Dirección de Supervisión Administrativa de Servicios Intermunicipales", "label": "ORGANIZATION"},
{"text": "932,80 euros", "label": "AMOUNT"},
{"text": "Unidad de Atención Ciudadana de la Comarca Interior", "label": "ORGANIZATION"},
{"text": "2018-05-15", "label": "DATE"},
{"text": "Patricia Giménez Domínguez", "label": "PERSON_NAME"},
{"text": "Servicio de Procedimientos Públicos de la Región Central", "label": "ORGANIZATION"},
{"text": "RES-2018-43300", "label": "REGISTRATION_NUMBER"},
{"text": "21 de mayo de 2018", "label": "DATE"},
{"text": "Calle de la Estación, 38, 6-2a, 08221 Terrassa", "label": "ADDRESS"},
{"text": "Patricia Giménez Domínguez", "label": "PERSON_NAME"},
{"text": "Tomás Romero Domínguez", "label": "PERSON_NAME"}
]""",
}

In [5]:
import re

def to_entities(text, items):
    # Convierte [{"text", "label"}] en entidades con posiciones start/end
    entities, used = [], set()
    for it in items:
        ent_text, label = it["text"], it["label"]
        pos = text.find(ent_text)
        # Si esa aparición ya está usada (mención repetida), buscamos la siguiente
        while pos != -1 and (pos, ent_text) in used:
            pos = text.find(ent_text, pos + 1)
        if pos == -1:
            print("No encontrada en el texto:", repr(ent_text), label)
            continue
        used.add((pos, ent_text))
        entities.append({
            "start": pos, "end": pos + len(ent_text),
            "text": ent_text, "label": label,
        })
    return entities


def parse_answer(raw):
    # Quita las marcas ```json ... ``` si el chat las ha puesto
    raw = re.sub(r"^```(?:json)?|```$", "", raw.strip(), flags=re.MULTILINE).strip()
    return json.loads(raw)


responses = []
for s in samples:
    if s["id"] not in LLM_ANSWERS:
        print("Sin respuesta todavía:", s["id"])
        break   # la evaluación necesita documentos consecutivos desde el primero
    try:
        items = parse_answer(LLM_ANSWERS[s["id"]])
    except json.JSONDecodeError as e:
        print("JSON inválido en", s["id"], "->", e)
        break
    responses.append({
        "id": s["id"],
        "text": s["text"],
        "entities": to_entities(s["text"], items),
        "language": s.get("language"),
    })

print(len(responses), "documentos procesados")
save_responses_to_jsonl(responses, OUTPUT_PATH)

Sin respuesta todavía: test-0002
1 documentos procesados


In [6]:
strict_results = evaluate_ner_strict(VALIDATION_PATH, OUTPUT_PATH)
relaxed_results = evaluate_ner_relaxed(VALIDATION_PATH, OUTPUT_PATH)


STRICT EVALUATION
Micro -> P: 1.00 | R: 1.00 | F1: 1.00 | TP: 41 | FP: 0 | FN: 0

ENTITY TYPE                    P      R     F1      TP      FP      FN
--------------------------------------------------------------------------------------------
PERSON_NAME                 1.00   1.00   1.00      10       0       0
ID_CARD_NUMBER              1.00   1.00   1.00       4       0       0
ADDRESS                     1.00   1.00   1.00       4       0       0
REGISTRATION_NUMBER         1.00   1.00   1.00       5       0       0
DATE                        1.00   1.00   1.00       8       0       0
AMOUNT                      1.00   1.00   1.00       4       0       0
ORGANIZATION                1.00   1.00   1.00       6       0       0

RELAXED EVALUATION - OCCURRENCES
Micro -> P: 1.00 | R: 1.00 | F1: 1.00 | TP: 41 | FP: 0 | FN: 0

ENTITY TYPE                    P      R     F1      TP      FP      FN
---------------------------------------------------------------------------------------